In [ ]:
import os
import sys

import cmocean
import h5py
import numpy as np
import torch
from matplotlib import pyplot as plt

_gap_aware_root = os.environ.get(
    "GAP_AWARE_ROOT",
    os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", "..", ".."))
)
if _gap_aware_root not in sys.path:
    sys.path.insert(0, _gap_aware_root)

from model_gap_aware.maskfeat import MaskFeatGapFiller
from esdc.src.evaluation.evaluate_gapfilling import ESDCGapFillingEvalDataset, find_checkpoint

# ── config ────────────────────────────────────────────────────────────────────
_esdc_gap_filling_training_dir = os.environ.get(
    "ESDC_GAP_FILLING_TRAINING_DIR",
    os.path.join(_gap_aware_root, "esdc", "data", "lst_gap_filling_training"),
)
h5_path    = os.path.join(_esdc_gap_filling_training_dir, "dataset.h5")
log_dir    = os.path.join(_gap_aware_root, "esdc", "lightning_logs/gapfilling_training")
checkpoint = None   # set to a .ckpt path to override auto-discovery
device     = "cpu"
n_samples  = 5
seed       = None

N_TIMESTEPS = 5
MIDDLE_T    = N_TIMESTEPS // 2   # index 2

# ESDC normalization: LST stored as (K − 180) / 180
LST_MIN, LST_MAX = 180.0, 360.0
CMAP = cmocean.cm.matter_r.copy()
VMIN, VMAX = 273.15, 323.15   # display range in Kelvin (0 °C – 50 °C)

# Aux variable display — colormaps, physical ranges, and display limits
# matching visualize_channels.ipynb
AUX_CHANNELS = [
    dict(name="Air temperature",  norm_range=(205.13, 311.35), cmap=cmocean.cm.matter_r, vmin=273.15, vmax=323.15),
    dict(name="LAI",              norm_range=(0.0, 14.447),    cmap=cmocean.cm.algae,    vmin=0,      vmax=4),
    dict(name="Surface moisture", norm_range=(0.007, 0.857),   cmap=cmocean.cm.dense,    vmin=0,      vmax=0.85),
]


def denorm_lst(arr):
    """Convert normalized [0, 1] LST back to Kelvin."""
    return arr * (LST_MAX - LST_MIN) + LST_MIN


# ── load model ────────────────────────────────────────────────────────────────
ckpt_path = find_checkpoint(log_dir, checkpoint)
print(f"Loading checkpoint: {ckpt_path}")
model = MaskFeatGapFiller.load_from_checkpoint(ckpt_path, map_location=device)
model.to(device)
model.eval()
PATCH_SIZE = model.patch_size

# ── patch-mask helpers ────────────────────────────────────────────────────────
def compute_gap_patch_mask(frame, patch_size):
    """frame: (C, H, W) or (H, W) — NaN at gaps. Returns (H, W) bool, True = patch masked."""
    if frame.ndim == 2:
        frame = frame[np.newaxis]
    C, H, W = frame.shape
    n_ph, n_pw = H // patch_size, W // patch_size
    r = frame.reshape(C, n_ph, patch_size, n_pw, patch_size)
    patch_has_nan = np.isnan(r).any(axis=(0, 2, 4))
    return np.repeat(np.repeat(patch_has_nan, patch_size, axis=0), patch_size, axis=1).astype(bool)


masked_cmap = CMAP.copy()
masked_cmap.set_bad("lightgray")

# ── dataset + reservoir sampling ──────────────────────────────────────────────
dataset = ESDCGapFillingEvalDataset(h5_path)
print(f"Dataset size: {len(dataset)} samples")

rng = np.random.default_rng(seed)
chosen_indices = sorted(
    rng.choice(len(dataset), size=min(n_samples, len(dataset)), replace=False).tolist()
)

# ── run inference and collect display data ────────────────────────────────────
samples      = []
total_gap_px = 0
total_err    = 0.0

with h5py.File(h5_path, "r") as f, torch.no_grad():
    for idx in chosen_indices:
        x_tensor, gt_tensor, gap_mask_tensor = dataset[idx]

        # Raw arrays from HDF5 for display (channels-last)
        x_raw = f["X_train"][idx].astype(np.float32)   # (T, H, W, C)
        y_raw = f["y_train"][idx]                       # (1, H, W, 1)

        x_in    = x_tensor.unsqueeze(0).to(device)          # (1, T, C, H, W)
        gt_in   = gt_tensor.unsqueeze(0).to(device)          # (1, 1, H, W)
        mask_in = gap_mask_tensor.unsqueeze(0).to(device)    # (1, H, W)

        pred, _ = model(x_in, gap_only_masking=True)         # (1, C, H, W)

        # Error over gap pixels with valid GT — channel 0 (LST) only
        gt_valid  = ~torch.isnan(gt_in)                      # (1, 1, H, W)
        gt_clean  = torch.nan_to_num(gt_in, nan=0.0)
        loss_mask = (mask_in.unsqueeze(1).bool() & gt_valid).float()  # (1, 1, H, W)

        pred_lst  = pred[:, :1]                              # (1, 1, H, W) LST channel
        err       = torch.abs(pred_lst - gt_clean)           # (1, 1, H, W)
        err_hw    = (err * loss_mask).squeeze()              # (H, W)

        n_gap        = loss_mask.sum().item()
        total_gap_px += n_gap
        total_err    += (err * loss_mask).sum().item()

        # Reconstruct full clean middle frame: non-gap from X, gap from y
        lst_norm   = x_raw[:, :, :, 0]                       # (T, H, W) normalized
        y_gap_norm = y_raw[0, :, :, 0].astype(np.float32)   # (H, W) NaN at non-gap px
        gap_np     = gap_mask_tensor.numpy().astype(bool)    # (H, W)
        gt_full    = np.where(gap_np, y_gap_norm, lst_norm[MIDDLE_T])  # (H, W)

        samples.append({
            "idx":      idx,
            "lst_norm": lst_norm,                             # (T, H, W) normalized
            "gt_full":  gt_full,                              # (H, W) full clean GT
            "gap_mask": gap_np,                               # (H, W) bool
            "pred_lst": pred_lst.squeeze().cpu().numpy(),     # (H, W) predicted LST
            "err_hw":   err_hw.cpu().numpy(),                 # (H, W) absolute error
            "x_raw":    x_raw,                                # (T, H, W, C) all channels
        })

overall_mae = total_err / max(total_gap_px, 1)
print(
    f"Mean MAE over gap pixels ({len(samples)} samples): "
    f"{overall_mae:.4f}  [{overall_mae * 180:.2f} K]"
)

# ── plot ──────────────────────────────────────────────────────────────────────
# Three rows per sample:
#   0 — input:    t0 | t1 | t2 (gappy) | t3 | t4 | GT | Prediction | Error
#   1 — aux:      air_temp | LAI | surface_moisture | LST-only mask | combined mask | aux-added mask | (empty) x2
#   2 — combined masked view: t0..t4 with all-channel gap mask | (empty) x3
N_COLS            = N_TIMESTEPS + 3   # 8
N_ROWS_PER_SAMPLE = 3
n_found           = len(samples)

fig, axes = plt.subplots(
    N_ROWS_PER_SAMPLE * n_found, N_COLS,
    figsize=(3.5 * N_COLS, 4.2 * N_ROWS_PER_SAMPLE * n_found),
    squeeze=False,
)

for row, s in enumerate(samples):
    row_in   = N_ROWS_PER_SAMPLE * row
    row_aux  = N_ROWS_PER_SAMPLE * row + 1
    row_mask = N_ROWS_PER_SAMPLE * row + 2

    lst_K  = denorm_lst(s["lst_norm"])   # (T, H, W) Kelvin
    gt_K   = denorm_lst(s["gt_full"])    # (H, W) Kelvin
    pred_K = denorm_lst(s["pred_lst"])   # (H, W) Kelvin
    err_K  = s["err_hw"] * 180.0         # (H, W) Kelvin

    # -- input row: time series + GT + prediction + error (unchanged) ----------
    for t in range(N_TIMESTEPS):
        ax    = axes[row_in][t]
        frame = np.ma.masked_invalid(lst_K[t])
        frac  = 100.0 * float(np.isnan(lst_K[t]).sum()) / lst_K[t].size
        label = " (gappy)" if t == MIDDLE_T else ""
        ax.imshow(frame, cmap=CMAP, vmin=VMIN, vmax=VMAX)
        ax.set_title(f"t={t}{label}\ngaps={frac:.1f}%", fontsize=7)
        ax.axis("off")

    ax_gt = axes[row_in][N_TIMESTEPS]
    ax_gt.imshow(np.ma.masked_invalid(gt_K), cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax_gt.set_title("GT (t=2)\nclean", fontsize=7)
    for spine in ax_gt.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("#1a7f1a")
        spine.set_linewidth(2)
    ax_gt.axis("off")

    ax_pred = axes[row_in][N_TIMESTEPS + 1]
    pred_merged_K = np.where(s["gap_mask"] | np.isnan(lst_K[MIDDLE_T]), pred_K, lst_K[MIDDLE_T])
    ax_pred.imshow(np.ma.masked_invalid(pred_merged_K), cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax_pred.set_title("Prediction (t=2)\ngap+NaN px merged", fontsize=7)
    for spine in ax_pred.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("#1a1a8f")
        spine.set_linewidth(2)
    ax_pred.axis("off")

    ax_err   = axes[row_in][N_TIMESTEPS + 2]
    err_disp = np.where(s["gap_mask"], err_K, np.nan)
    vmax_err = np.nanpercentile(err_disp, 95) if np.any(~np.isnan(err_disp)) else 5.0
    im       = ax_err.imshow(err_disp, cmap="viridis", vmin=0, vmax=vmax_err)
    coverage = 100.0 * float(s["gap_mask"].sum()) / max(s["gap_mask"].size, 1)
    ax_err.set_title(f"Error (gap px) [K]\ncoverage={coverage:.1f}%", fontsize=7)
    ax_err.axis("off")
    plt.colorbar(im, ax=ax_err, fraction=0.046, pad=0.04)

    axes[row_in][0].set_ylabel(f"sample {s['idx']}", fontsize=8, labelpad=4)
    axes[row_in][0].yaxis.set_visible(True)

    # -- aux row: aux channels at t=2  +  three mask panels -------------------
    x_mid = s["x_raw"][MIDDLE_T]   # (H, W, C)

    for a, ch_cfg in enumerate(AUX_CHANNELS):
        ax      = axes[row_aux][a]
        lo, hi  = ch_cfg["norm_range"]
        ch_norm = x_mid[:, :, a + 1]              # normalized [0, 1]
        ch_phys = ch_norm * (hi - lo) + lo        # physical units
        nan_pct = 100.0 * float(np.isnan(ch_norm).mean())
        cmap_ch = ch_cfg["cmap"].copy()
        cmap_ch.set_bad("lightgray")
        ax.imshow(np.ma.masked_invalid(ch_phys), cmap=cmap_ch,
                  vmin=ch_cfg["vmin"], vmax=ch_cfg["vmax"])
        ax.set_title(f"{ch_cfg['name']}\nt=2 | NaN={nan_pct:.1f}%", fontsize=7)
        ax.axis("off")

    # Compute masks at t=2: LST-only, all-channel combined, difference
    lst_ch        = x_mid[:, :, 0]            # (H, W)
    all_chs       = x_mid.transpose(2, 0, 1)  # (C, H, W)
    lst_only_mask = compute_gap_patch_mask(lst_ch,  PATCH_SIZE)
    combined_mask = compute_gap_patch_mask(all_chs, PATCH_SIZE)
    aux_added     = combined_mask & ~lst_only_mask

    for col_i, (mask, title, cmap_m) in enumerate([
        (lst_only_mask, "LST-only mask\n(t=2)",  "gray_r"),
        (combined_mask, "Combined mask\n(t=2)",  "gray_r"),
        (aux_added,     "Aux-added mask\n(t=2)", "Reds"),
    ]):
        ax  = axes[row_aux][3 + col_i]
        pct = 100.0 * float(mask.mean())
        ax.imshow(mask.astype(float), cmap=cmap_m, vmin=0, vmax=1)
        ax.set_title(f"{title}\nmasked={pct:.1f}%", fontsize=7)
        ax.axis("off")

    for col in range(6, N_COLS):
        axes[row_aux][col].axis("off")
    axes[row_aux][0].set_ylabel("aux channels\n& masks", fontsize=7, labelpad=4)
    axes[row_aux][0].yaxis.set_visible(True)

    # -- combined masked-view row: all channels drive the patch mask -----------
    for t in range(N_TIMESTEPS):
        ax           = axes[row_mask][t]
        all_chs_t    = s["x_raw"][t].transpose(2, 0, 1)   # (C, H, W)
        pixel_mask   = compute_gap_patch_mask(all_chs_t, PATCH_SIZE)
        frame_masked = lst_K[t].copy()
        frame_masked[pixel_mask] = np.nan
        ax.imshow(np.ma.masked_invalid(frame_masked), cmap=masked_cmap, vmin=VMIN, vmax=VMAX)
        pct   = 100.0 * float(pixel_mask.mean())
        label = " (gappy)" if t == MIDDLE_T else ""
        ax.set_title(f"t={t}{label}\nmasked={pct:.1f}%", fontsize=7)
        ax.axis("off")
    for col in range(N_TIMESTEPS, N_COLS):
        axes[row_mask][col].axis("off")
    axes[row_mask][0].set_ylabel("combined\nmasked view", fontsize=7, labelpad=4)
    axes[row_mask][0].yaxis.set_visible(True)

fig.suptitle(
    f"Gap-filling training model — ESDC training samples  ({n_found} samples)  |  "
    f"MAE (gap px) = {overall_mae:.4f} [{overall_mae * 180:.2f} K]\n"
    "Green border = GT clean  |  Blue border = pred merged with input (NaN→model)  |  "
    "Error map: |pred−GT| at synthetic gap pixels [Kelvin, viridis]\n"
    "Aux row: aux channels at t=2 (physical units) + LST-only / combined / aux-added patch masks  |  "
    "Masked-view row: lightgray = patch masked by encoder (all channels combined)",
    fontsize=9,
    y=1.005,
)
plt.tight_layout()
plt.show()